In [20]:
import numpy as np
import pandas as pd

In [21]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

In [22]:
df = pd.read_csv("../../dataset/raw/emails.csv", low_memory=False)
print(f"Shape: {df.shape}")
print(f"\nNull counts:\n{df.isnull().sum()}")

Shape: (123389, 27)

Null counts:
Co_Ref                                      0
Time_to_Renewal                             0
crm_accreditation_completed             21035
crm_timely_completion                   21035
crm_progress_towards_accreditation      21035
crm_delays_in_accreditation             21035
crm_contractor_suggested_leave          21035
crm_contractor_engagement               21035
crm_contractor_sentiment                21035
crm_contractor_sentiment_score          21035
crm_dts_or_ssip_mentioned               21035
crm_customer_payment_intention          21035
crm_competitors_mentioned               11155
crm_membership_level                    11155
crm_platform_issues_raised              11155
crm_agent_chased_contractor             11155
crm_agent_chase_count                   11155
crm_accreditation_issues                11155
crm_membership_overdue                  11155
crm_auto_renewal_status                 11155
crm_dissatisified_with_renewal_price    11155


In [23]:
df = df[df["Time_to_Renewal"] == "pre_renewal"]

## 1 — Understand the three null groups

The dataset has three distinct null-count tiers, indicating rows generated by different pipeline batches where certain feature extraction was skipped:

- **3 155 nulls** — early-batch columns (accreditation, sentiment, etc.)
- **1 518 nulls** — mid-batch columns (membership, agent chase, etc.)
- **1 630 nulls** — late-batch columns (complaints, refunds, etc.)

Since all affected columns are categorical flags extracted from call/email text, the safest imputation is `"Not Discussed"` — it accurately represents _"information not captured in that interaction"_.

In [24]:
# ── Column groups by null-count tier ────────────────────────────────────────

group_3155 = [
    'crm_accreditation_completed', 'crm_timely_completion',
    'crm_progress_towards_accreditation', 'crm_delays_in_accreditation',
    'crm_contractor_suggested_leave', 'crm_contractor_engagement',
    'crm_contractor_sentiment', 'crm_contractor_sentiment_score',
    'crm_dts_or_ssip_mentioned', 'crm_customer_payment_intention',
]

group_1518 = [
    'crm_competitors_mentioned', 'crm_membership_level',
    'crm_platform_issues_raised', 'crm_agent_chased_contractor',
    'crm_agent_chase_count', 'crm_accreditation_issues',
    'crm_membership_overdue', 'crm_auto_renewal_status',
    'crm_dissatisified_with_renewal_price',
]

group_1630 = [
    'crm_customer_complained', 'crm_refund_mentioned',
    'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned',
]

# Verify coverage
all_null_cols = group_3155 + group_1518 + group_1630
print("Columns with nulls:", len(all_null_cols))

Columns with nulls: 24


## 2 — Helper: normalise dirty free-text values to canonical labels

The LLM that generated these CRM flags occasionally returned fragments of explanatory text instead of a clean `Yes / No / Not Discussed`. We normalise these by keyword matching.

In [25]:
def normalise_flag(value, valid_values=('Yes', 'No', 'Not Discussed')):
    """
    Maps a raw string to one of the canonical valid_values.
    Rules (in priority order):
      1. NaN / actual NaN  → None  (will be imputed later)
      2. Placeholder tokens → None
      3. 'Not applicable'   → 'Not Discussed'
      4. Exact match        → as-is
      5. Starts with valid  → strip to valid
      6. Contains 'yes'     → 'Yes'
      7. Contains 'no'      → 'No'
      8. Fallback           → 'Not Discussed'
    """
    if pd.isna(value):
        return None

    s = str(value).strip()

    # Placeholder / unanswered
    placeholders = ['[yes/no]', '[yes/no/not discussed]', '[yes/no/not discussed]']
    if s.lower() in placeholders:
        return None

    # Not-applicable variants → Not Discussed
    if s.lower().startswith('not applicable'):
        return 'Not Discussed'

    # Exact match
    if s in valid_values:
        return s

    # Starts-with match (e.g. "Yes, the agent...")
    for v in valid_values:
        if s.startswith(v):
            return v

    # Keyword scan (case-insensitive)
    sl = s.lower()
    if 'not discussed' in sl:
        return 'Not Discussed'
    if sl.startswith('yes') or ' yes' in sl:
        return 'Yes'
    if sl.startswith('no') or 'dissatisfied' in sl or 'negative' in sl or 'indicating' in sl:
        return 'Yes'   # "indicating dissatisfaction" is effectively Yes for negative-sentiment cols

    # Fallback
    return 'Not Discussed'

## 3 — Clean binary / ternary flag columns

In [26]:
# Columns that should only hold Yes / No / Not Discussed
ync_cols = [
    'crm_accreditation_completed', 'crm_timely_completion',
    'crm_progress_towards_accreditation', 'crm_delays_in_accreditation',
    'crm_contractor_suggested_leave', 'crm_contractor_engagement',
    'crm_dts_or_ssip_mentioned', 'crm_customer_payment_intention',
    'crm_competitors_mentioned', 'crm_platform_issues_raised',
    'crm_agent_chased_contractor', 'crm_accreditation_issues',
    'crm_membership_overdue', 'crm_dissatisified_with_renewal_price',
    'crm_customer_complained', 'crm_refund_mentioned',
    'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned',
]

for col in ync_cols:
    df[col] = df[col].apply(normalise_flag)

# Impute remaining NaNs with 'Not Discussed'
for col in ync_cols:
    df[col] = df[col].fillna('Not Discussed')

# Quick sanity check
for col in ync_cols:
    unique = set(df[col].unique())
    unexpected = unique - {'Yes', 'No', 'Not Discussed'}
    if unexpected:
        print(f"  ⚠  {col}: unexpected values → {unexpected}")
print("Binary/ternary flag columns cleaned ✓")

Binary/ternary flag columns cleaned ✓


## 4 — Clean `crm_contractor_sentiment`

In [27]:
SENTIMENT_MAP = {
    'Neutral':                              'Neutral',
    'Not Discussed':                        'Not Discussed',
    'Satisfied':                            'Satisfied',
    'Dissatisfied':                         'Dissatisfied',
    'Initially Dissatisfied, later Neutral':'Neutral',   # resolved → Neutral
    'Yes':                                  'Satisfied',  # mislabelled
    'No':                                   'Dissatisfied',
}

def clean_sentiment(val):
    if pd.isna(val):
        return 'Not Discussed'
    s = str(val).strip()
    if s in SENTIMENT_MAP:
        return SENTIMENT_MAP[s]
    sl = s.lower()
    if 'dissatisfied' in sl or 'negative' in sl:
        return 'Dissatisfied'
    if 'satisfied' in sl or 'positive' in sl:
        return 'Satisfied'
    if 'neutral' in sl:
        return 'Neutral'
    return 'Not Discussed'

df['crm_contractor_sentiment'] = df['crm_contractor_sentiment'].apply(clean_sentiment)
print(df['crm_contractor_sentiment'].value_counts())

crm_contractor_sentiment
Not Discussed    10492
Neutral           8613
Satisfied         2257
Dissatisfied      1504
Name: count, dtype: int64


## 5 — Clean `crm_contractor_sentiment_score`

Valid range is **0–100** (numeric). A handful of rows contain the text labels `'Dissatisfied'`, `'Neutral'` or partial fragments — we map these to their numeric equivalents (20 and 50 respectively).

In [28]:
SCORE_TEXT_MAP = {
    'Not Discussed': np.nan,
    'Dissatisfied':  20.0,
    'Neutral':       50.0,
}

def clean_score(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s in SCORE_TEXT_MAP:
        return SCORE_TEXT_MAP[s]
    try:
        num = float(s)
        return num if 0 <= num <= 100 else np.nan
    except ValueError:
        return np.nan

df['crm_contractor_sentiment_score'] = (
    df['crm_contractor_sentiment_score'].apply(clean_score)
)

# Impute NaN with median (robust to outliers)
median_score = df['crm_contractor_sentiment_score'].median()
df['crm_contractor_sentiment_score'] = (
    df['crm_contractor_sentiment_score'].fillna(median_score)
)
print(f"Sentiment score — median used for imputation: {median_score}")
print(df['crm_contractor_sentiment_score'].describe())

Sentiment score — median used for imputation: 50.0
count    22866.000000
mean        51.442950
std         13.714776
min          0.000000
25%         50.000000
50%         50.000000
75%         50.000000
max        100.000000
Name: crm_contractor_sentiment_score, dtype: float64


## 6 — Clean `crm_agent_chase_count`

This column should be numeric but contains:
- Two separate `0` buckets (4 290 + 4 069) — both are genuine zeros
- Textual descriptions like `'Multiple'`, `'A few'`, `'Several'`
- Long free-text sentences

Strategy: extract digits if present; map vague words to conservative estimates; NaN (no data) → 0.

In [29]:
import re

VAGUE_CHASE_MAP = {
    'a few':        3,
    'several':      4,
    'multiple':     5,
    'many':         6,
    'numerous':     6,
    'more than 7':  8,
    'a number of':  4,
}

def clean_chase_count(val):
    if pd.isna(val):
        return 0  # no data → agent did not chase
    s = str(val).strip().lower()

    # Explicit 'not discussed' / 'not applicable' / 'not specified' → 0
    if any(t in s for t in ['not discussed', 'not applicable', 'not specified']):
        return 0

    # Try direct numeric parse first
    try:
        return int(float(s))
    except ValueError:
        pass

    # Extract first integer from string (e.g. "2 (at least...)")
    digits = re.findall(r'\b(\d+)\b', s)
    if digits:
        return int(digits[0])

    # Map vague words
    for keyword, count in VAGUE_CHASE_MAP.items():
        if keyword in s:
            return count

    return 0  # safe fallback

df['crm_agent_chase_count'] = df['crm_agent_chase_count'].apply(clean_chase_count)
df['crm_agent_chase_count'] = df['crm_agent_chase_count'].astype(int)
print(df['crm_agent_chase_count'].value_counts().sort_index())

crm_agent_chase_count
0     9906
1     8125
2     3819
3      833
4      112
5       47
6       11
7        7
8        1
10       2
11       1
13       1
30       1
Name: count, dtype: int64


## 7 — Clean `crm_auto_renewal_status`

Should be an integer flag: `0` = off, `1` = on, `2` = pending/other. Free-text fragments are mapped to `0`.

In [30]:
def clean_auto_renewal(val):
    if pd.isna(val):
        return 0  # not mentioned → treat as off / not applicable
    s = str(val).strip()
    try:
        v = int(float(s))
        return v if v in (0, 1, 2) else 0
    except ValueError:
        # Free-text remnant → 0
        return 0

df['crm_auto_renewal_status'] = df['crm_auto_renewal_status'].apply(clean_auto_renewal)
df['crm_auto_renewal_status'] = df['crm_auto_renewal_status'].astype(int)
print(df['crm_auto_renewal_status'].value_counts())

crm_auto_renewal_status
0    21500
2     1226
1      140
Name: count, dtype: int64


## 8 — Clean `crm_membership_level`

Consolidate ~18 raw variants into 5 canonical levels.

In [31]:
def clean_membership_level(val):
    if pd.isna(val):
        return 'Not Discussed'
    s = str(val).strip().lower()

    if 'not discussed' in s or 'not accredited' in s:
        return 'Not Discussed'
    if 'in progress' in s:
        return 'In Progress'
    if 'gold' in s or 'premier' in s or 'prem' in s:
        return 'Premier'
    if 'silver' in s:
        return 'Standard'
    if 'accredited' in s:   # covers 'accredited', 'accredited (gold level)'
        return 'Accredited'
    if 'standard' in s or 'members only' in s or 'membership' in s:
        return 'Standard'
    if 'express' in s:
        return 'Standard'
    return 'Not Discussed'

df['crm_membership_level'] = df['crm_membership_level'].apply(clean_membership_level)
print(df['crm_membership_level'].value_counts())

crm_membership_level
Not Discussed    9962
Accredited       7153
In Progress      5621
Standard          122
Premier             8
Name: count, dtype: int64


## 9 — Clean `crm_dts_or_ssip_mentioned`

A few rows hold numeric scores (0, 30, 50) left over from a mislabelled pipeline run. Map any non-Yes/No value to `Not Discussed`.

In [32]:
df['crm_dts_or_ssip_mentioned'] = df['crm_dts_or_ssip_mentioned'].apply(
    lambda v: normalise_flag(v, valid_values=('Yes', 'No', 'Not Discussed'))
).fillna('Not Discussed')

print(df['crm_dts_or_ssip_mentioned'].value_counts())

crm_dts_or_ssip_mentioned
No               15478
Yes               4229
Not Discussed     3159
Name: count, dtype: int64


## 10 — Final null check & dtype consolidation

In [33]:
remaining_nulls = df.isnull().sum()
print("Remaining nulls after cleaning:")
print(remaining_nulls[remaining_nulls > 0] if remaining_nulls.any() else "None ✓")

# Convert all clean categorical string columns to pandas Categorical
cat_cols = ync_cols + ['crm_contractor_sentiment', 'crm_membership_level']
for col in cat_cols:
    df[col] = pd.Categorical(df[col])

# Ensure numeric columns are correct dtype
df['crm_contractor_sentiment_score'] = df['crm_contractor_sentiment_score'].astype(float)
df['crm_agent_chase_count']           = df['crm_agent_chase_count'].astype(int)
df['crm_auto_renewal_status']         = df['crm_auto_renewal_status'].astype(int)


Remaining nulls after cleaning:
None ✓


In [34]:
print(f"Final shape: {df.shape}")
print("\nCategorical column distributions:")
for col in cat_cols:
    print(f"\n{col}:")
    counts = df[col].value_counts()
    pct    = (counts / len(df) * 100).round(1)
    print(pd.concat([counts, pct.rename('%')], axis=1).to_string())

Final shape: (22866, 27)

Categorical column distributions:

crm_accreditation_completed:


                             count     %
crm_accreditation_completed             
Not Discussed                13877  60.7
No                            6742  29.5
Yes                           2247   9.8

crm_timely_completion:
                       count     %
crm_timely_completion             
Not Discussed          20979  91.7
No                      1474   6.4
Yes                      413   1.8

crm_progress_towards_accreditation:
                                    count     %
crm_progress_towards_accreditation             
Not Discussed                       16945  74.1
Yes                                  5343  23.4
No                                    578   2.5

crm_delays_in_accreditation:
                             count     %
crm_delays_in_accreditation             
No                           14988  65.5
Yes                           4464  19.5
Not Discussed                 3414  14.9

crm_contractor_suggested_leave:
                                count     %
crm_con

In [35]:
print("\nNumeric column stats:")
print(df[['crm_contractor_sentiment_score', 'crm_agent_chase_count',
          'crm_auto_renewal_status']].describe())


Numeric column stats:
       crm_contractor_sentiment_score  crm_agent_chase_count  \
count                    22866.000000           22866.000000   
mean                        51.442950               0.837138   
std                         13.714776               0.936037   
min                          0.000000               0.000000   
25%                         50.000000               0.000000   
50%                         50.000000               1.000000   
75%                         50.000000               1.000000   
max                        100.000000              30.000000   

       crm_auto_renewal_status  
count             22866.000000  
mean                  0.113356  
std                   0.455795  
min                   0.000000  
25%                   0.000000  
50%                   0.000000  
75%                   0.000000  
max                   2.000000  


In [36]:
df.to_csv("../../dataset/preprocessed/emails_preprocessed.csv", index=False)